# Signal documentation ingestion

Unity Catalog Volume に置かれた PDF、DOCX、PPTX、XLSX を読み込み、ページ・スライド・シート単位のテキストを Delta テーブルへ保存します。ウィジェットを環境に合わせて設定してから、上から順に実行してください。

In [ ]:
%pip install --quiet pypdf python-docx python-pptx openpyxl

In [ ]:
# Volume パスと出力先を環境に合わせて変更してください。
dbutils.widgets.text("signal_docs_path", "/Volumes/<catalog>/<schema>/docs", "Signal docs Volume path")
dbutils.widgets.text("target_table", "<catalog>.<schema>.blf_signal_doc_sections", "Target Delta table")
dbutils.widgets.text("semantic_model_endpoint", "", "Model Serving endpoint (optional)")

SIGNAL_DOCS_PATH = dbutils.widgets.get("signal_docs_path").rstrip("/")
TARGET_TABLE = dbutils.widgets.get("target_table").strip()
SEMANTIC_MODEL_ENDPOINT = dbutils.widgets.get("semantic_model_endpoint").strip()

if not SIGNAL_DOCS_PATH.startswith("/Volumes/"):
    raise ValueError("signal_docs_path must be a Unity Catalog Volume path such as /Volumes/catalog/schema/volume/path")
if "<" in TARGET_TABLE or TARGET_TABLE.count(".") != 2:
    raise ValueError("target_table must be a three-part Unity Catalog table name: catalog.schema.table")

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone

import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.types import (
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

SECTIONS_SCHEMA = StructType(
    [
        StructField("_source_file", StringType(), nullable=False),
        StructField("_file_mtime", TimestampType()),
        StructField("_file_size_bytes", LongType()),
        StructField("_ingested_at", TimestampType()),
        StructField("doc_type", StringType(), nullable=False),
        StructField("section_number", IntegerType(), nullable=False),
        StructField("section_label", StringType()),
        StructField("text", StringType()),
    ]
)


def _local_path(spark_path: str) -> str:
    if spark_path.startswith("dbfs:/Volumes/"):
        return spark_path[len("dbfs:") :]
    if spark_path.startswith("dbfs:/"):
        return "/dbfs/" + spark_path[len("dbfs:/") :]
    return spark_path


def _extract_pdf_sections(path: str):
    import pypdf

    reader = pypdf.PdfReader(path)
    return [(i, f"Page {i}", page.extract_text() or "") for i, page in enumerate(reader.pages, 1)]


def _extract_docx_sections(path: str):
    import docx

    document = docx.Document(path)
    return [(1, "Document", "\n".join(p.text for p in document.paragraphs if p.text))]


def _extract_pptx_sections(path: str):
    from pptx import Presentation

    presentation = Presentation(path)
    return [
        (
            i,
            f"Slide {i}",
            "\n".join(
                shape.text_frame.text for shape in slide.shapes if shape.has_text_frame and shape.text_frame.text
            ),
        )
        for i, slide in enumerate(presentation.slides, 1)
    ]


def _extract_xlsx_sections(path: str):
    import openpyxl

    workbook = openpyxl.load_workbook(path, data_only=True, read_only=True)
    sections = []
    for i, sheet in enumerate(workbook.worksheets, 1):
        lines = ["\t".join(str(cell) for cell in row if cell is not None) for row in sheet.iter_rows(values_only=True)]
        sections.append((i, f"Sheet '{sheet.title}'", "\n".join(line for line in lines if line)))
    return sections


EXTRACTORS = {
    "pdf": ("PDF", _extract_pdf_sections),
    "docx": ("DOCX", _extract_docx_sections),
    "pptx": ("PPTX", _extract_pptx_sections),
    "xlsx": ("XLSX", _extract_xlsx_sections),
}


def _parse_doc_batch(iterator):
    for batch_df in iterator:
        rows = []
        for _, row in batch_df.iterrows():
            spark_path = str(row["_source_file"])
            local_path = _local_path(spark_path)
            extension = local_path.rsplit(".", 1)[-1].lower()
            extractor = EXTRACTORS.get(extension)
            if extractor is None:
                continue
            doc_type, extract = extractor
            try:
                parsed_sections = extract(local_path)
            except Exception as exc:
                print(f"[signal_docs] failed to parse {spark_path!r}: {exc}")
                continue
            for section_number, section_label, text in parsed_sections:
                rows.append(
                    {
                        "_source_file": spark_path,
                        "_file_mtime": row.get("_file_mtime"),
                        "_file_size_bytes": row.get("_file_size_bytes"),
                        "_ingested_at": datetime.now(timezone.utc),
                        "doc_type": doc_type,
                        "section_number": section_number,
                        "section_label": section_label,
                        "text": text,
                    }
                )
        if rows:
            yield pd.DataFrame(rows)

In [ ]:
# binaryFile は Volume 配下を再帰的に走査します。対象拡張子だけを抽出関数へ渡します。
files = (
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load(SIGNAL_DOCS_PATH)
    .where(F.lower(F.col("path")).rlike(r"\.(pdf|docx|pptx|xlsx)$"))
    .select(
        F.col("path").alias("_source_file"),
        F.col("modificationTime").alias("_file_mtime"),
        F.col("length").alias("_file_size_bytes"),
    )
)

sections = files.mapInPandas(_parse_doc_batch, schema=SECTIONS_SCHEMA)
if SEMANTIC_MODEL_ENDPOINT:
    endpoint = SEMANTIC_MODEL_ENDPOINT.replace("'", "''")
    prompt = (
        "Summarize this automotive signal documentation excerpt in 1-3 sentences. "
        "Call out signal names, their physical meaning, and units if present. Excerpt: "
    )
    sections = sections.withColumn(
        "semantic_summary",
        F.expr(f"ai_query('{endpoint}', concat('{prompt}', text))"),
    )
else:
    sections = sections.withColumn("semantic_summary", F.lit(None).cast(StringType()))

# 同じファイル版・セクションを再実行しても重複させないための安定キー。
sections = sections.withColumn(
    "section_id",
    F.sha2(
        F.concat_ws(
            "||",
            F.col("_source_file"),
            F.coalesce(F.col("_file_mtime").cast("string"), F.lit("")),
            F.coalesce(F.col("_file_size_bytes").cast("string"), F.lit("")),
            F.col("section_number").cast("string"),
        ),
        256,
    ),
)

In [ ]:
import re


def _quoted_table_name(name: str) -> str:
    parts = name.split(".")
    if len(parts) != 3 or not all(re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", part) for part in parts):
        raise ValueError("target_table must contain only three simple identifiers: catalog.schema.table")
    return ".".join(f"`{part}`" for part in parts)


quoted_target = _quoted_table_name(TARGET_TABLE)
if not spark.catalog.tableExists(TARGET_TABLE):
    (
        sections.write.format("delta")
        .mode("errorifexists")
        .option("delta.autoOptimize.optimizeWrite", "true")
        .saveAsTable(TARGET_TABLE)
    )
else:
    sections.createOrReplaceTempView("_signal_doc_sections_source")
    spark.sql(f"""
        MERGE INTO {quoted_target} AS target
        USING _signal_doc_sections_source AS source
        ON target.section_id = source.section_id
        WHEN NOT MATCHED THEN INSERT *
    """)

display(spark.table(TARGET_TABLE).orderBy(F.col("_ingested_at").desc()).limit(100))